In [1]:
from pathlib import Path
import pandas as pd

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
DATA_PATH = Path("../../data/processed/dataset_candidates_rag.csv")
OUTPUT_PATH = Path("../../data/processed/dataset_candidates_rag_bertopic_clusters.csv")
TOPIC_INFO_PATH = Path("../../data/processed/bertopic_topic_info.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

In [3]:
# Change sep=";" to sep="," if your CSV uses commas.
df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

Dataset shape: (7451, 9)
Columns:
['triggerTitle', 'triggerChannelTitle', 'actionTitle', 'actionChannelTitle', 'title', 'desc', 'trigger_class', 'action_class', 'rule_scope']


,triggerTitle,triggerChannelTitle,actionTitle,actionChannelTitle,title,desc,trigger_class,action_class,rule_scope
0,Item added to your Shopping List,Amazon Alexa,Send an SMS,Android SMS,"If item added to your To Do List, then send me an SMS at 56937","If item added to your To Do List, then send me an SMS at 56937",smart_home,general_digital,smart_home_related
1,Item added to your Shopping List,Amazon Alexa,Append to a text file,Dropbox,"Add new items to your Alexa Shopping List, append a file in Dropbox","If a new item is added to your Alexa Shopping List, then append to a text file in Dropbox",smart_home,general_digital,smart_home_related
2,Item deleted on your Shopping List,Amazon Alexa,Send an email,Gmail,If shopping list item deleted send email,"If I delete an item in Alexa App, send me an updated email",smart_home,general_digital,smart_home_related
3,Your Timer goes off,Amazon Alexa,Send a notification from the IFTTT app,Notifications,"If Timer Goes Off, A IF Notification Is Sent","When your Echo's timer goes off, a notification will be sent to your iOS device.",smart_home,general_digital,smart_home_related
4,Your Alarm goes off,Amazon Alexa,Send a notification from the IFTTT app,Notifications,Receive Notification on Alarm,"If Alexa alarm goes off, then send a notification to phone",smart_home,general_digital,smart_home_related


In [4]:
# Columns used by BERTopic to build the clustering text.
# These are the actual textual fields of the rule.
CLUSTERING_COLUMNS = [
    "triggerTitle",
    "triggerChannelTitle",
    "actionTitle",
    "actionChannelTitle",
    "title",
    "desc",
]

# These columns are kept for inspection, but they are NOT used for clustering.
# We do not want BERTopic to cluster based on these already existing labels.
METADATA_COLUMNS = [
    "trigger_class",
    "action_class",
    "rule_scope",
]

REQUIRED_COLUMNS = CLUSTERING_COLUMNS + METADATA_COLUMNS

# Check required columns
missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}")

# Clean text columns
for col in REQUIRED_COLUMNS:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Remove rows whose rule_scope contains "smart_home_related"
n_before = len(df)

df_cluster = df[
    ~df["rule_scope"].str.contains(
        "smart_home_related",
        case=False,
        na=False
    )
].copy()

n_after = len(df_cluster)
n_removed = n_before - n_after

print(f"Rows before filtering: {n_before}")
print(f"Rows removed: {n_removed}")
print(f"Rows after filtering: {n_after}")

df_cluster[REQUIRED_COLUMNS].head()

Rows before filtering: 7451
Rows removed: 3994
Rows after filtering: 3457


,triggerTitle,triggerChannelTitle,actionTitle,actionChannelTitle,title,desc,trigger_class,action_class,rule_scope
39,Smart Home/Away,ecobee,Quick add event,Google Calendar,Log Home/Away on Calendar,"IF SmartHome/Away then logged it on Google Calendar, this can help if you have CCTV installed and want to know if a motion is detected and you can check your CCTV on specific day/time",smart_home,context,context_aware_smart_home
103,The security panel triggers an alert,HomeControl Flex,Send message to a circle,Life360,Notify my family if the security system triggers an alert.,Receive a notification if the security system triggers an alert.,smart_home,context,context_aware_smart_home
129,Battery is low,Nest Protect,Voice announcement,Ubi,Nest Protect Battery low Ubi will let user know.,If the battery is low on Nest Protect Ubi can tell you.,smart_home,smart_home,core_smart_home
140,New Motion Detected,Ring,Start recording,Arlo,"If new Motion detected at Front Door, then Arlo Starts Recording","If new Motion detected at Front Door, then Arlo Starts Recording",smart_home,smart_home,core_smart_home
158,Any new motion,SmartThings,Send my car a notification,BMW Labs,Receive a dashboard notification if your SmartThings detects motion,Get a dashboard notification if your SmartThings device senses motion.,smart_home,smart_home,core_smart_home


In [5]:
def build_topic_text(row):
    return (
        f"When {row['triggerTitle']} from {row['triggerChannelTitle']}, "
        f"do {row['actionTitle']} using {row['actionChannelTitle']}. "
        f"{row['title']}. {row['desc']}"
    )


df_cluster["topic_text"] = df_cluster.apply(build_topic_text, axis=1)

docs = df_cluster["topic_text"].tolist()

df_cluster[["topic_text"]].head()

,topic_text
39,"When Smart Home/Away from ecobee, do Quick add event using Google Calendar. Log Home/Away on Calendar. IF SmartHome/Away then logged it on Google Calendar, this can help if you have CCTV installed..."
103,"When The security panel triggers an alert from HomeControl Flex, do Send message to a circle using Life360. Notify my family if the security system triggers an alert.. Receive a notification if th..."
129,"When Battery is low from Nest Protect, do Voice announcement using Ubi. Nest Protect Battery low Ubi will let user know.. If the battery is low on Nest Protect Ubi can tell you."
140,"When New Motion Detected from Ring, do Start recording using Arlo. If new Motion detected at Front Door, then Arlo Starts Recording. If new Motion detected at Front Door, then Arlo Starts Recording"
158,"When Any new motion from SmartThings, do Send my car a notification using BMW Labs. Receive a dashboard notification if your SmartThings detects motion. Get a dashboard notification if your SmartT..."


In [6]:
# This model converts each rule text into a semantic embedding.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# UMAP reduces embedding dimensionality before clustering.
# Since we removed smart_home_related rows, the dataset is more focused.
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

# HDBSCAN creates clusters and can mark noisy/unclear rules as topic -1.
# Slightly smaller clusters are allowed because the filtered dataset is cleaner.
hdbscan_model = HDBSCAN(
    min_cluster_size=35,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

custom_stop_words = list(ENGLISH_STOP_WORDS.union({
    "rule", "rules",
    "action", "actions",
    "trigger", "triggers",
    "original",
    "smart", "home",
    "automation", "automations",
    "system", "executes", "execute",
    "main", "important", "grouping", "similar",
    "title", "description", "descriptions",
    "text",

    # Technical words from the generated topic text
    "trigger_channel",
    "action_channel",
    "channel",
    "channels",

    # Generic platform / automation words
    "app",
    "ifttt",
    "send",
    "sent",
    "receive",
    "received",
    "specific",

    # Generic words introduced by build_topic_text
    "using",
}))

vectorizer_model = CountVectorizer(
    stop_words=custom_stop_words,
    ngram_range=(1, 3),
    min_df=4,
    max_df=0.6,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)

ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True,
    reduce_frequent_words=True,
)

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=15,
    calculate_probabilities=False,
    verbose=True,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
docs = df_cluster["topic_text"].tolist()

# BERTopic assigns one topic_id to each rule.
topics, _ = topic_model.fit_transform(docs)

df_cluster["topic_id"] = topics

n_topics_including_outliers = df_cluster["topic_id"].nunique()
n_outliers = (df_cluster["topic_id"] == -1).sum()
outlier_ratio = n_outliers / len(df_cluster)

print("Number of topic IDs, including outliers:", n_topics_including_outliers)
print("Number of real topics, excluding outliers:", n_topics_including_outliers - (1 if -1 in df_cluster["topic_id"].unique() else 0))
print("Number of outliers, topic_id = -1:", n_outliers)
print(f"Outlier ratio: {outlier_ratio:.2%}")

df_cluster[["topic_text", "topic_id"]].head()

2026-05-22 03:28:57,673 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/109 [00:00<?, ?it/s]

2026-05-22 03:29:20,591 - BERTopic - Embedding - Completed ✓
2026-05-22 03:29:20,593 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-22 03:29:42,754 - BERTopic - Dimensionality - Completed ✓
2026-05-22 03:29:42,755 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-22 03:29:42,811 - BERTopic - Cluster - Completed ✓
2026-05-22 03:29:42,818 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-22 03:29:45,080 - BERTopic - Representation - Completed ✓


Number of topic IDs, including outliers: 40
Number of real topics, excluding outliers: 39
Number of outliers, topic_id = -1: 475
Outlier ratio: 13.74%


,topic_text,topic_id
39,"When Smart Home/Away from ecobee, do Quick add event using Google Calendar. Log Home/Away on Calendar. IF SmartHome/Away then logged it on Google Calendar, this can help if you have CCTV installed...",-1
103,"When The security panel triggers an alert from HomeControl Flex, do Send message to a circle using Life360. Notify my family if the security system triggers an alert.. Receive a notification if th...",7
129,"When Battery is low from Nest Protect, do Voice announcement using Ubi. Nest Protect Battery low Ubi will let user know.. If the battery is low on Nest Protect Ubi can tell you.",27
140,"When New Motion Detected from Ring, do Start recording using Arlo. If new Motion detected at Front Door, then Arlo Starts Recording. If new Motion detected at Front Door, then Arlo Starts Recording",36
158,"When Any new motion from SmartThings, do Send my car a notification using BMW Labs. Receive a dashboard notification if your SmartThings detects motion. Get a dashboard notification if your SmartT...",6


In [8]:
# This table summarizes the topics found by BERTopic.
#
# Columns:
# - Topic: topic id
# - Count: number of rules in the topic
# - Name: automatic topic name generated from representative words

topic_info = topic_model.get_topic_info()

display(topic_info.head(50))

# Save topic overview for the filtered clustering version
TOPIC_INFO_FILTERED_PATH = "bertopic_topic_info_filtered.csv"

topic_info.to_csv(TOPIC_INFO_FILTERED_PATH, index=False)

print("Topic overview saved at:", TOPIC_INFO_FILTERED_PATH)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,475,-1_use smartthings_detects_automatically turn_run homeseer event,"[use smartthings, detects, automatically turn, run homeseer event, blink lights, blink lights philips, blink, auto, event nest, homeseer event]","[When Button press from Button widget, do Start Cool Mode using tado Air Conditioning. Boost air conditioning to __ ÃÂÃÂÃÂÃÂ°. Want to give your air conditioning a quick boost? Now it's jus..."
1,0,160,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"[presence detected smartthings, detected smartthings, alarm activated, turn automatically, smartthings turn, notifications, alerts, alert, smartthings, sensor]","[When Opened from SmartThings, do Start recording using Oco Camera. Start record video if door opened. Applet starts record video from Oco camera if your SmatThings multipurpose Sensor opened, Whe..."
2,1,157,1_use wink_set ecobee_automatic_turn device,"[use wink, set ecobee, automatic, turn device, ecobee, ecobee sleep, transition ecobee, automatic classic, enabled, pulse]","[When New daily movement logged from UP by Jawbone, do Quick add event using Google Calendar. Step Motivation. text/email a buddy(s) when you hit your step goal for the day., When Sleep goal achie..."
3,2,153,2_automatically turn_philips hue turn_turn switch_change light,"[automatically turn, philips hue turn, turn switch, change light, philips hue lights, turn lights philips, turn light, applet turn, hue turn, relay]","[When Sunset from Weather Underground, do Power on device using Energenie Mi|Home. Switch your Mi|Home adapter ON everyday at sunset. The socket will switch on at sunset each day, When Sunset from..."
4,3,150,3_change light_turn light_lamp_blink lights philips,"[change light, turn light, lamp, blink lights philips, color philips hue, change light color, blink lights, change color philips, turn red, blink]","[When Current condition changes to from Weather Underground, do Call a function using Particle. Turn ""take umbrella"" warning light off if weather changes to ""clear"". This is one in 4 Applets used ..."
5,4,148,4_turn heater_insight switch_turn ac_plug turn,"[turn heater, insight switch, turn ac, plug turn, turn wemo switch, heater, wemo insight switch, turn link, turn link plug, air conditioner]","[When Current temperature drops below from Weather Underground, do Turn on using WeMo Smart Plug. Turn on heater. Choose the weather channel in your city, When Current temperature rises above from..."
6,5,137,5_set thermostat_add event google_ecobee_event starts google,"[set thermostat, add event google, ecobee, event starts google, scheduled, event google, add event, thermostat set thermostat, starts google, set temperature]","[When Event from search starts from Google Calendar, do Set oven to sabbath mode using GE Appliances Cooking. Turn on Sabbath mode using Google Calendar. Set a ""Sabbath Starts"" event on your calen..."
7,6,111,6_start activity harmony_activity harmony turn_activity harmony_start activity,"[start activity harmony, activity harmony turn, activity harmony, start activity, harmony turn, harmony, turn philips hue, turn tv, sound link siren, link siren]","[When Any new motion from WeMo Motion, do Turn lights on using LIFX. Turn on your lights when new motion is detected. This Recipe will turn your Lifx lights on whenever your WeMo motion detects mo..."
8,7,108,7_siren_alarm turn_motion sensor_alarm change,"[siren, alarm turn, motion sensor, alarm change, location start, alarm turns, profile transition ecobee, alert, setting, area location start]","[When First family member arrives at a specific place from Life360, do Disarm all modes using Scout Alarm. Disarm Scout Alarm when first family member arrives to a place. Family geofence disarming..."
9,8,96,8_amazon alexa start_alexa start_tell alexa turn_phrase amazon alexa,"[amazon alexa start, alexa start, tell alexa turn, phrase amazon alexa, alexa turn,

Topic overview saved at: bertopic_topic_info_filtered.csv


In [9]:
# Add BERTopic's automatic topic name to each row.
# This makes the exported CSV easier to inspect.
#

topic_names = topic_info.set_index("Topic")["Name"].to_dict()

df_cluster["topic_name"] = df_cluster["topic_id"].map(topic_names)

df_cluster[
    [
        "topic_id",
        "topic_name",
        "topic_text",
        "trigger_class",
        "action_class",
        "rule_scope",
    ]
].head()

,topic_id,topic_name,topic_text,trigger_class,action_class,rule_scope
39,-1,-1_use smartthings_detects_automatically turn_run homeseer event,"When Smart Home/Away from ecobee, do Quick add event using Google Calendar. Log Home/Away on Calendar. IF SmartHome/Away then logged it on Google Calendar, this can help if you have CCTV installed...",smart_home,context,context_aware_smart_home
103,7,7_siren_alarm turn_motion sensor_alarm change,"When The security panel triggers an alert from HomeControl Flex, do Send message to a circle using Life360. Notify my family if the security system triggers an alert.. Receive a notification if th...",smart_home,context,context_aware_smart_home
129,27,27_light alarm_hue blink lights_blink device_blinks,"When Battery is low from Nest Protect, do Voice announcement using Ubi. Nest Protect Battery low Ubi will let user know.. If the battery is low on Nest Protect Ubi can tell you.",smart_home,smart_home,core_smart_home
140,36,36_philips hue turn_hue turn lights_change hue lights_turn hue light,"When New Motion Detected from Ring, do Start recording using Arlo. If new Motion detected at Front Door, then Arlo Starts Recording. If new Motion detected at Front Door, then Arlo Starts Recording",smart_home,smart_home,core_smart_home
158,6,6_start activity harmony_activity harmony turn_activity harmony_start activity,"When Any new motion from SmartThings, do Send my car a notification using BMW Labs. Receive a dashboard notification if your SmartThings detects motion. Get a dashboard notification if your SmartT...",smart_home,smart_home,core_smart_home


In [13]:
# Save the filtered dataset plus:
# - topic_text
# - topic_id
# - topic_name
#
# trigger_class, action_class, and rule_scope are still present in the final CSV,
# but they were NOT used to create the clusters.
#
# Rows whose rule_scope contains "smart_home_related" were removed before clustering.

OUTPUT_FILTERED_PATH = "dataset_candidates_rag_bertopic_clusters_no_smart_home_related.csv"

df_cluster.to_csv(OUTPUT_FILTERED_PATH, index=False)

print("Clustered filtered dataset saved at:", OUTPUT_FILTERED_PATH)
print("Saved shape:", df_cluster.shape)

Clustered filtered dataset saved at: dataset_candidates_rag_bertopic_clusters_no_smart_home_related.csv
Saved shape: (3457, 12)


In [11]:
# Quick summary of cluster sizes.
# Topic -1 contains outliers.

cluster_summary = (
    df_cluster.groupby(["topic_id", "topic_name"])
    .size()
    .reset_index(name="n_rules")
    .sort_values("n_rules", ascending=False)
)

display(cluster_summary)

,topic_id,topic_name,n_rules
0,-1,-1_use smartthings_detects_automatically turn_run homeseer event,475
1,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,160
2,1,1_use wink_set ecobee_automatic_turn device,157
3,2,2_automatically turn_philips hue turn_turn switch_change light,153
4,3,3_change light_turn light_lamp_blink lights philips,150
5,4,4_turn heater_insight switch_turn ac_plug turn,148
6,5,5_set thermostat_add event google_ecobee_event starts google,137
7,6,6_start activity harmony_activity harmony turn_activity harmony_start activity,111
8,7,7_siren_alarm turn_motion sensor_alarm change,108
9,8,8_amazon alexa start_alexa start_tell alexa turn_phrase amazon alexa,96


In [12]:
# Change this value to inspect a different cluster.

CLUSTER_ID = 0

cluster_df = df_cluster[df_cluster["topic_id"] == CLUSTER_ID].copy()

print("Cluster size:", len(cluster_df))
print("Topic words:")
print(topic_model.get_topic(CLUSTER_ID))

display(
    cluster_df[
        [
            "topic_id",
            "topic_name",
            "topic_text",
            "triggerChannelTitle",
            "triggerTitle",
            "actionChannelTitle",
            "actionTitle",
            "title",
            "desc",
            "trigger_class",
            "action_class",
            "rule_scope",
        ]
    ].sample(min(20, len(cluster_df)), random_state=42)
)

Cluster size: 160
Topic words:
[('presence detected smartthings', np.float32(0.48270223)), ('detected smartthings', np.float32(0.42264295)), ('alarm activated', np.float32(0.40615934)), ('turn automatically', np.float32(0.37528965)), ('smartthings turn', np.float32(0.37351418)), ('notifications', np.float32(0.36412418)), ('alerts', np.float32(0.35080346)), ('alert', np.float32(0.3407184)), ('smartthings', np.float32(0.32022)), ('sensor', np.float32(0.31467146))]


,topic_id,topic_name,topic_text,triggerChannelTitle,triggerTitle,actionChannelTitle,actionTitle,title,desc,trigger_class,action_class,rule_scope
3908,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When Every day of the week at from Date & Time, do Aim your camera using iSecurity+. Point my camera somewhere interesting. You got some privacy by pointing your camera at the wall using another A...",Date & Time,Every day of the week at,iSecurity+,Aim your camera,Point my camera somewhere interesting,You got some privacy by pointing your camera at the wall using another AppletNow use this Applet to automatically point it back into the room at the time you usually go to work(Works with Kodak CF...,context,smart_home,context_aware_smart_home
3933,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When Every day at from Date & Time, do Start recording using Manything. Turn on Manything when you're away. Set the days and times that you need Manything to recordDon't forget to leave Manything ...",Date & Time,Every day at,Manything,Start recording,Turn on Manything when you're away,Set the days and times that you need Manything to recordDon't forget to leave Manything open at the record screenWhen Manything starts recording you can login and watch your streams live and remot...,context,smart_home,context_aware_smart_home
5875,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When Any new motion from SmartThings, do Record video using SkyBell HD. Start Skybell video record when motion is detected. using Smartthings motion sensor to trigger Skybell video, since motion s...",SmartThings,Any new motion,SkyBell HD,Record video,Start Skybell video record when motion is detected,"using Smartthings motion sensor to trigger Skybell video, since motion sense on Skybell is limited and will not detect motion in front of garage.",smart_home,smart_home,core_smart_home
2674,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When First family member arrives at a specific place from Life360, do Change camera mode using Withings Home. Family Location Withings Home - Turn camera off when first family member arrives home....",Life360,First family member arrives at a specific place,Withings Home,Change camera mode,Family Location Withings Home - Turn camera off when first family member arrives home,Family Location Withings Home - Turn camera off when first family member arrives home,context,smart_home,context_aware_smart_home
3720,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When Button press from Button widget, do Control camera shutter using Somfy Protect. Close the privacy shutter. Close the Myfox Security Camera privacy shutter",Button widget,Button press,Somfy Protect,Control camera shutter,Close the privacy shutter,Close the Myfox Security Camera privacy shutter,smart_home,smart_home,core_smart_home
1871,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When You enter an area from Location, do Control camera shutter using Somfy Protect. When you arrive home, close the Myfox Security Camera privacy shutter. When you arrive home, your Myfox Securit...",Location,You enter an area,Somfy Protect,Control camera shutter,"When you arrive home, close the Myfox Security Camera privacy shutter","When you arrive home, your Myfox Security Camera's privacy shutter will me closed to help give you piece of mind and privacy.",context,smart_home,context_aware_smart_home
3855,0,0_presence detected smartthings_detected smartthings_alarm activated_turn automatically,"When Every day at from Date & Time, do Start recording using Camio. Start recording for your cameras every morning. Every morning, Camio will start recording for your cameras.",Date & Time,Every day at,Camio,Start recording,Start recording for your cameras every m